# Preprocessing & Feature Engineering Roadmap
## 1. Feature Analysis and Feature Engineering
## 2. Data Preprocessing
## 3. Train-Test Split
## 4. Save Dataset for Training

In [ ]:
# 1. Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os


# Text preprocessing
import re
from sklearn.model_selection import train_test_split

# Vectorization methods
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import BertTokenizer, BertModel
import torch

# Feature analysis and selection
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import pearsonr


# For saving processed data
import pickle
import joblib

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:

# 2. Load the dataset
email_df = pd.read_csv('datasets/processed/cleaned_email_dataset.csv')
email_df

In [ ]:
# Drop unnecessary columns
email_df = email_df.drop(columns=['source_dataset', 'urls_original'])

print("Columns after dropping:")
print(email_df.columns.tolist())
print(f"\nDataset shape: {email_df.shape}")
print("\nRemaining columns:")
print(email_df.info())

## 1. Feature Analysis and Feature Engineering

##### Working with sender feature

In [ ]:
def extract_sender_components(sender):
    """
    Extract sender_name, sender_email, and sender_domain from sender string
    Format: "Name <email@domain.com>" or "email@domain.com"
    """
    if pd.isna(sender) or sender.strip() == '':
        return pd.Series({'sender_name': None, 'sender_email': None, 'sender_domain': None})

    sender = sender.strip()

    # Pattern: "Name <email@domain.com>"
    pattern = r'(.+?)\s*<(.+?)>'
    match = re.search(pattern, sender)

    if match:
        sender_name = match.group(1).strip()
        sender_email = match.group(2).strip()
    else:
        # No name, just email
        sender_name = None
        sender_email = sender.strip()

    # Remove any remaining < or > from email
    if sender_email:
        sender_email = sender_email.replace('<', '').replace('>', '').strip()

    # Extract domain from email
    if sender_email and '@' in sender_email:
        sender_domain = sender_email.split('@')[-1].strip()
    else:
        sender_domain = None

    return pd.Series({
        'sender_name': sender_name,
        'sender_email': sender_email,
        'sender_domain': sender_domain
    })

# Apply extraction
email_df[['sender_name', 'sender_email', 'sender_domain']] = email_df['sender'].apply(extract_sender_components)

print("\nSender components extracted!")
print("\nSample results:")
print(email_df[['sender', 'sender_name', 'sender_email', 'sender_domain']].head(10))

print("\nMissing values check:")
print(email_df[['sender_name', 'sender_email', 'sender_domain']].isnull().sum())

#### CREATE SENDER FEATURES

In [ ]:
# 1. sender_missing - Binary flag if sender is missing
email_df['sender_missing'] = (email_df['sender'].isna() |
                               (email_df['sender'].str.strip() == '')).astype(int)

# 2. sender_name_length - Length of sender name (0 if no name)
email_df['sender_name_length'] = email_df['sender_name'].fillna('').str.len()

# 3. email_local_length - Length of email before @ symbol
def get_email_local_length(email):
    if pd.isna(email) or email.strip() == '':
        return 0
    if '@' in email:
        return len(email.split('@')[0])
    return 0

email_df['email_local_length'] = email_df['sender_email'].apply(get_email_local_length)

# 4. domain_length - Length of domain name
email_df['domain_length'] = email_df['sender_domain'].fillna('').str.len()

# 5. domain_has_numbers - Binary flag if domain contains digits
email_df['domain_has_numbers'] = email_df['sender_domain'].fillna('').str.contains(r'\d', regex=True).astype(int)

# 6. sender_is_noreply - Binary flag if email starts with noreply
def is_noreply(email):
    if pd.isna(email) or email.strip() == '':
        return 0
    email_lower = email.lower()
    return int(bool(re.match(r'^no[-_]?reply', email_lower)))

email_df['sender_is_noreply'] = email_df['sender_email'].apply(is_noreply)

# 7. consecutive_digit_length - Longest sequence of consecutive digits in email local part
def get_consecutive_digit_length(email):
    if pd.isna(email) or email.strip() == '':
        return 0
    local_part = email.split('@')[0] if '@' in email else email
    digit_sequences = re.findall(r'\d+', local_part)
    if digit_sequences:
        return max(len(seq) for seq in digit_sequences)
    return 0

email_df['consecutive_digit_length'] = email_df['sender_email'].apply(get_consecutive_digit_length)

# 8. special_char_count - Count of special characters in email local part
def count_special_chars(email):
    if pd.isna(email) or email.strip() == '':
        return 0
    local_part = email.split('@')[0] if '@' in email else email
    special_chars = re.findall(r'[^a-zA-Z0-9.]', local_part)
    return len(special_chars)

email_df['special_char_count'] = email_df['sender_email'].apply(count_special_chars)


#### INTRINSIC DOMAIN FEATURES (No lookup needed)

In [ ]:
# 9. domain_entropy - Measure of randomness in domain name (higher = more random/suspicious)
import math

def calculate_entropy(text):
    """Calculate Shannon entropy - measures randomness of text"""
    if not text or len(text) == 0:
        return 0
    # Count character frequencies
    freq = {}
    for char in text:
        freq[char] = freq.get(char, 0) + 1
    # Calculate entropy
    entropy = 0
    text_len = len(text)
    for count in freq.values():
        probability = count / text_len
        entropy -= probability * math.log2(probability)
    return entropy

def get_domain_entropy(domain):
    if pd.isna(domain) or domain.strip() == '':
        return 0
    # Remove TLD for entropy calculation (focus on main domain)
    domain_parts = domain.split('.')
    if len(domain_parts) >= 2:
        main_domain = '.'.join(domain_parts[:-1])  # Everything except TLD
    else:
        main_domain = domain
    return calculate_entropy(main_domain.lower())

email_df['domain_entropy'] = email_df['sender_domain'].apply(get_domain_entropy)

# 10. domain_vowel_ratio - Ratio of vowels in domain (random domains have unusual ratios)
def get_vowel_ratio(domain):
    if pd.isna(domain) or domain.strip() == '':
        return 0
    domain_clean = re.sub(r'[^a-zA-Z]', '', domain.lower())  # Keep only letters
    if len(domain_clean) == 0:
        return 0
    vowels = 'aeiou'
    vowel_count = sum(1 for char in domain_clean if char in vowels)
    return vowel_count / len(domain_clean)

email_df['domain_vowel_ratio'] = email_df['sender_domain'].apply(get_vowel_ratio)

# 11. domain_consonant_ratio - Ratio of consonants in domain
def get_consonant_ratio(domain):
    if pd.isna(domain) or domain.strip() == '':
        return 0
    domain_clean = re.sub(r'[^a-zA-Z]', '', domain.lower())  # Keep only letters
    if len(domain_clean) == 0:
        return 0
    consonants = 'bcdfghjklmnpqrstvwxyz'
    consonant_count = sum(1 for char in domain_clean if char in consonants)
    return consonant_count / len(domain_clean)

email_df['domain_consonant_ratio'] = email_df['sender_domain'].apply(get_consonant_ratio)

#### LOOKUP-BASED FEATURES (Requires saving statistics)

In [ ]:
# 12. domain_frequency - How many times this domain appears in dataset
domain_freq = email_df['sender_domain'].value_counts().to_dict()
email_df['domain_frequency'] = email_df['sender_domain'].map(domain_freq).fillna(0).astype(int)

# 13. is_rare_domain - Binary flag if domain appears 3 or fewer times
email_df['is_rare_domain'] = (email_df['domain_frequency'] <= 3).astype(int)

# 14. Extract TLD for lookup features
def get_domain_tld(domain):
    if pd.isna(domain) or domain.strip() == '':
        return None
    parts = domain.split('.')
    if len(parts) >= 2:
        return parts[-1].lower()
    return None

email_df['domain_tld'] = email_df['sender_domain'].apply(get_domain_tld)

# 15. tld_frequency - How many times this TLD appears in dataset
tld_freq = email_df['domain_tld'].value_counts().to_dict()
email_df['tld_frequency'] = email_df['domain_tld'].map(tld_freq).fillna(0).astype(int)

# 16. tld_phishing_ratio - Ratio of phishing emails for this TLD
tld_phishing_stats = email_df.groupby('domain_tld')['label'].agg(['sum', 'count'])
tld_phishing_ratio_map = (tld_phishing_stats['sum'] / tld_phishing_stats['count']).to_dict()
email_df['tld_phishing_ratio'] = email_df['domain_tld'].map(tld_phishing_ratio_map).fillna(0.5)  # Default 0.5 for unknown

#### CREATE LOOKUP TABLES FOR INFERENCE

In [ ]:
# Lookup table 1: Domain frequency
lookup_domain_frequency = domain_freq

# Lookup table 2: TLD frequency
lookup_tld_frequency = tld_freq

# Lookup table 3: TLD phishing ratio
lookup_tld_phishing_ratio = tld_phishing_ratio_map

print(f"Domain frequency lookup table created: {len(lookup_domain_frequency)} unique domains")
print(f"TLD frequency lookup table created: {len(lookup_tld_frequency)} unique TLDs")
print(f"TLD phishing ratio lookup table created: {len(lookup_tld_phishing_ratio)} unique TLDs")

# Display sample from lookup tables
print("\nSample from lookup tables:")
print("\nTop 10 most common domains:")
print(pd.Series(lookup_domain_frequency).sort_values(ascending=False).head(10))
print("\nTop 10 most common TLDs:")
print(pd.Series(lookup_tld_frequency).sort_values(ascending=False).head(10))
print("\nTLD phishing ratios (sample):")
print(pd.Series(lookup_tld_phishing_ratio).sort_values(ascending=False).head(10))

print("\nAll sender features created!")

# Display results
print("\nFeature summary:")
feature_cols = ['sender_missing', 'sender_name_length', 'email_local_length',
                'domain_length', 'domain_has_numbers', 'sender_is_noreply',
                'consecutive_digit_length', 'special_char_count',
                'domain_entropy', 'domain_vowel_ratio', 'domain_consonant_ratio',
                'domain_frequency', 'is_rare_domain', 'tld_frequency', 'tld_phishing_ratio']

print(email_df[feature_cols].describe())

print("\nSample rows with features:")
display_cols = ['sender', 'sender_missing', 'email_local_length', 'domain_length',
                'consecutive_digit_length', 'special_char_count', 'domain_entropy',
                'domain_frequency', 'is_rare_domain', 'tld_phishing_ratio']
print(email_df[display_cols].head(10))

print(f"\nDataset shape: {email_df.shape}")
print(f"Total sender features created: {len(feature_cols)}")

#### SAVE LOOKUP TABLES TO JSON

In [ ]:
# Create directory if it doesn't exist
os.makedirs('lookup_tables', exist_ok=True)

# Save lookup_domain_frequency
with open('lookup_tables/domain_frequency.json', 'w') as f:
    json.dump(lookup_domain_frequency, f, indent=2)
print(f"✓ Saved: lookup_tables/domain_frequency.json ({len(lookup_domain_frequency)} domains)")

# Save lookup_tld_frequency
with open('lookup_tables/tld_frequency.json', 'w') as f:
    json.dump(lookup_tld_frequency, f, indent=2)
print(f"✓ Saved: lookup_tables/tld_frequency.json ({len(lookup_tld_frequency)} TLDs)")

# Save lookup_tld_phishing_ratio
with open('lookup_tables/tld_phishing_ratio.json', 'w') as f:
    json.dump(lookup_tld_phishing_ratio, f, indent=2)
print(f"✓ Saved: lookup_tables/tld_phishing_ratio.json ({len(lookup_tld_phishing_ratio)} TLDs)")

# Also save combined lookup tables as pickle (for pipeline compatibility)
import pickle

lookup_tables_combined = {
    'domain_frequency': lookup_domain_frequency,
    'tld_frequency': lookup_tld_frequency,
    'tld_phishing_ratio': lookup_tld_phishing_ratio
}

with open('lookup_tables/lookup_tables.pkl', 'wb') as f:
    pickle.dump(lookup_tables_combined, f)
print(f"✓ Saved: lookup_tables/lookup_tables.pkl (combined pickle file)")


In [ ]:
email_df

# Sender Feature Engineering

## Feature Categories

### 1. Basic Sender Features (Always Computable)
These features can be calculated for any email without external data:

- **`sender_missing`**: Binary flag (0/1) indicating if sender field is empty
  - *Purpose*: Phishing emails sometimes have missing sender info

- **`sender_name_length`**: Character count of sender's display name
  - *Purpose*: Very short or very long names can indicate suspicious emails

- **`email_local_length`**: Length of email address before @ symbol
  - *Purpose*: Phishing often uses long random strings (e.g., "dwthaidomainnamesm@...")

- **`domain_length`**: Length of domain name after @ symbol
  - *Purpose*: Suspicious domains are often unusually long

- **`domain_has_numbers`**: Binary flag if domain contains digits
  - *Purpose*: Legitimate domains rarely have numbers

- **`sender_is_noreply`**: Binary flag if email starts with "noreply"
  - *Purpose*: Phishing often impersonates automated systems

- **`consecutive_digit_length`**: Longest sequence of consecutive digits in email local part
  - *Purpose*: Random number sequences (like "iplines1983") indicate auto-generated phishing emails

- **`special_char_count`**: Count of special characters (_, -, |, etc.) in email local part
  - *Purpose*: Excessive special characters indicate randomly generated emails

### 2. Intrinsic Domain Features (No Lookup Required)
These analyze the domain structure itself and work for new/unseen domains:

- **`domain_entropy`**: Shannon entropy measuring randomness of domain name (0-5 scale)
  - *Purpose*: Random character sequences = high entropy = suspicious
  - *Example*: "google" (low entropy) vs "xk9zm2pq" (high entropy)

- **`domain_vowel_ratio`**: Proportion of vowels in domain (0-1 scale)
  - *Purpose*: Real words have natural vowel patterns; random strings don't
  - *Example*: "amazon" (0.50) vs "bcdfgh" (0.00)

- **`domain_consonant_ratio`**: Proportion of consonants in domain (0-1 scale)
  - *Purpose*: Complement to vowel ratio for detecting unpronounceable domains

### 3. Lookup-Based Features (Requires Training Statistics)
These features use statistics calculated from training data:

- **`domain_frequency`**: Number of times this domain appears in training dataset
  - *Purpose*: Legitimate companies send multiple emails; phishing domains are one-time use
  - *Training*: Count occurrences in dataset
  - *Inference*: Lookup domain in saved map; if not found → frequency = 0 (new/suspicious)

- **`is_rare_domain`**: Binary flag if domain appears ≤ 3 times in training data
  - *Purpose*: Rare domains are more likely to be phishing
  - *Depends on*: domain_frequency

- **`tld_frequency`**: Number of times this TLD (.com, .org, etc.) appears in training dataset
  - *Purpose*: Common TLDs (.com, .org) vs rare TLDs (.tk, .xyz)
  - *Training*: Count TLD occurrences
  - *Inference*: Lookup TLD in saved map; if not found → frequency = 0

- **`tld_phishing_ratio`**: Proportion of emails with this TLD that are phishing (0-1 scale)
  - *Purpose*: Some TLDs are heavily used for phishing
  - *Example*: If .xyz has 80% phishing rate in training → ratio = 0.80
  - *Training*: Calculate (phishing_count / total_count) per TLD
  - *Inference*: Lookup TLD in saved map; if not found → use default 0.5 (unknown = 50% risk)

---

## How Lookup Tables Work

### Training Phase (This Notebook):
1. Calculate statistics from entire dataset
2. Create lookup dictionaries:
   - `lookup_domain_frequency`: {domain → count}
   - `lookup_tld_frequency`: {tld → count}
   - `lookup_tld_phishing_ratio`: {tld → phishing_ratio}
3. Save these dictionaries as pickle/joblib files

### Inference Phase (Future - Real-time Email Processing):
1. Load saved lookup tables
2. For new email:
   - Extract domain (e.g., "gmail.com")
   - Look up in `lookup_domain_frequency`
     - If found: use saved count
     - If NOT found: use 0 (new domain = potentially suspicious)
   - Extract TLD (e.g., "com")
   - Look up in `lookup_tld_frequency` and `lookup_tld_phishing_ratio`
     - If found: use saved values
     - If NOT found: use defaults (0 for frequency, 0.5 for phishing ratio)
3. Calculate all intrinsic features normally (no lookup needed)
4. Combine all features and pass to model

### Key Insight:
- **Intrinsic features** (entropy, vowel ratio, etc.) → Always computable for ANY email
- **Lookup features** (frequency, phishing ratio) → Use training statistics as reference
- **Unseen domains/TLDs** → Treated as suspicious (frequency=0) which helps detect new phishing attempts

---

## Feature Importance for Phishing Detection

**High Impact Features:**
- `tld_phishing_ratio` - Directly indicates TLD risk
- `domain_entropy` - Random domains are highly suspicious
- `consecutive_digit_length` - Long digit sequences indicate auto-generation
- `domain_frequency` - One-time domains are suspicious

**Medium Impact Features:**
- `email_local_length` - Very long local parts are suspicious
- `domain_vowel_ratio` - Unpronounceable domains are suspicious
- `special_char_count` - Excessive special chars indicate randomness

**Supporting Features:**
- `sender_missing`, `sender_is_noreply`, `domain_has_numbers` - Additional signals

## SUBJECT FEATURES

In [ ]:
# Handle missing subjects - fill with empty string for processing
email_df['subject'] = email_df['subject'].fillna('')

# 1. subject_length - character count
email_df['subject_length'] = email_df['subject'].str.len()

# 2. subject_word_count - number of words
email_df['subject_word_count'] = email_df['subject'].str.split().str.len().fillna(0).astype(int)

# 3. subject_missing - binary flag if subject is empty
email_df['subject_missing'] = (email_df['subject'].str.strip() == '').astype(int)

# 4. subject_exclamation_count - count of !
email_df['subject_exclamation_count'] = email_df['subject'].str.count('!')

# 5. subject_question_count - count of ?
email_df['subject_question_count'] = email_df['subject'].str.count(r'\?')  # FIXED: escaped the ?

# 6. subject_special_char_ratio - ratio of special characters to total characters
def get_special_char_ratio(text):
    if pd.isna(text) or len(text) == 0:
        return 0
    special_chars = re.findall(r'[^a-zA-Z0-9\s]', text)
    return len(special_chars) / len(text)

email_df['subject_special_char_ratio'] = email_df['subject'].apply(get_special_char_ratio)

# 7. subject_avg_word_length - average characters per word
def get_avg_word_length(text):
    if pd.isna(text) or text.strip() == '':
        return 0
    words = text.split()
    if len(words) == 0:
        return 0
    return sum(len(word) for word in words) / len(words)

email_df['subject_avg_word_length'] = email_df['subject'].apply(get_avg_word_length)

# 8. subject_entropy - Shannon entropy (measure of randomness)
def calculate_entropy(text):
    """Calculate Shannon entropy - measures randomness of text"""
    if not text or len(text) == 0:
        return 0
    # Count character frequencies
    freq = {}
    for char in text.lower():
        if char != ' ':  # Ignore spaces
            freq[char] = freq.get(char, 0) + 1
    # Calculate entropy
    entropy = 0
    text_len = len([c for c in text if c != ' '])
    if text_len == 0:
        return 0
    for count in freq.values():
        probability = count / text_len
        entropy -= probability * math.log2(probability)
    return entropy

email_df['subject_entropy'] = email_df['subject'].apply(calculate_entropy)


# Display results
print("\nSubject Feature Summary:")
subject_features = ['subject_length', 'subject_word_count', 'subject_missing',
                    'subject_exclamation_count', 'subject_question_count',
                    'subject_special_char_ratio', 'subject_avg_word_length', 'subject_entropy']

print(email_df[subject_features].describe())

print("\nSample rows:")
print(email_df[['subject', 'subject_length', 'subject_word_count', 'subject_exclamation_count',
                'subject_entropy']].head(10))

print(f"\nTotal subject features: {len(subject_features)}")

## BODY + URL FEATURES

In [ ]:
# Comprehensive URL pattern covering ALL common URL formats
url_patterns = [
    r'https?://[^\s<>\"\'\)]+',           # http:// or https://
    r'ftp://[^\s<>\"\'\)]+',              # ftp://
    r'ftps://[^\s<>\"\'\)]+',             # ftps://
    r'sftp://[^\s<>\"\'\)]+',             # sftp://
    r'www\.[^\s<>\"\'\)]+',               # www.
    r'file://[^\s<>\"\'\)]+',             # file://
    r'ssh://[^\s<>\"\'\)]+',              # ssh://
    r'telnet://[^\s<>\"\'\)]+',           # telnet://
    r'git://[^\s<>\"\'\)]+',              # git://
    r'svn://[^\s<>\"\'\)]+',              # svn://
    r'mailto:[^\s<>\"\'\)]+',             # mailto:
    r'news:[^\s<>\"\'\)]+',               # news:
    r'nntp://[^\s<>\"\'\)]+',             # nntp://
    r'irc://[^\s<>\"\'\)]+',              # irc://
    r'webcal://[^\s<>\"\'\)]+',           # webcal://
]

In [ ]:
# Handle missing body - fill with empty string for processing
email_df['body'] = email_df['body'].fillna('')

# 1. body_length - character count
email_df['body_length'] = email_df['body'].str.len()

# 2. body_word_count - number of words
email_df['body_word_count'] = email_df['body'].str.split().str.len().fillna(0).astype(int)

# 3. body_to_subject_length_ratio - body length / subject length
def get_body_to_subject_ratio(row):
    if row['subject_length'] == 0:
        return 0 if row['body_length'] == 0 else float('inf')
    return row['body_length'] / row['subject_length']

email_df['body_to_subject_length_ratio'] = email_df.apply(get_body_to_subject_ratio, axis=1)
# Replace infinity with a large number for practical use
email_df['body_to_subject_length_ratio'] = email_df['body_to_subject_length_ratio'].replace(float('inf'), 10000)

# 4. body_url_count - count actual URLs in body
def count_urls(text):
    if pd.isna(text) or text.strip() == '':
        return 0
    # Use the comprehensive url_patterns list
    total_urls = 0
    for pattern in url_patterns:
        urls = re.findall(pattern, text)
        total_urls += len(urls)
    return total_urls

email_df['body_url_count'] = email_df['body'].apply(count_urls)

# 5. body_url_density - URLs per 100 words
def get_url_density(row):
    if row['body_word_count'] == 0:
        return 0
    return (row['body_url_count'] / row['body_word_count']) * 100

email_df['body_url_density'] = email_df.apply(get_url_density, axis=1)

# 6. body_exclamation_count - count of !
email_df['body_exclamation_count'] = email_df['body'].str.count('!')

# 7. body_exclamation_density - ! per 100 characters
def get_exclamation_density(row):
    if row['body_length'] == 0:
        return 0
    return (row['body_exclamation_count'] / row['body_length']) * 100

email_df['body_exclamation_density'] = email_df.apply(get_exclamation_density, axis=1)

# 8. body_question_count - count of ?
email_df['body_question_count'] = email_df['body'].str.count(r'\?')  # FIXED: escaped the ?

# 9. body_avg_word_length - average characters per word
email_df['body_avg_word_length'] = email_df['body'].apply(get_avg_word_length)

# 10. body_entropy - Shannon entropy
email_df['body_entropy'] = email_df['body'].apply(calculate_entropy)

# 11. body_unique_word_ratio - unique words / total words
def get_unique_word_ratio(text):
    if pd.isna(text) or text.strip() == '':
        return 0
    words = text.lower().split()
    if len(words) == 0:
        return 0
    unique_words = len(set(words))
    return unique_words / len(words)

email_df['body_unique_word_ratio'] = email_df['body'].apply(get_unique_word_ratio)

# 12. body_line_count - number of lines (newlines)
email_df['body_line_count'] = email_df['body'].str.count('\n') + 1


# Display results
print("\nBody Feature Summary:")
body_features = ['body_length', 'body_word_count', 'body_to_subject_length_ratio',
                 'body_url_count', 'body_url_density', 'body_exclamation_count',
                 'body_exclamation_density', 'body_question_count', 'body_avg_word_length',
                 'body_entropy', 'body_unique_word_ratio', 'body_line_count']

print(email_df[body_features].describe())

print("\nSample rows:")
print(email_df[['body_length', 'body_word_count', 'body_url_count', 'body_url_density',
                'body_exclamation_count', 'body_entropy', 'body_unique_word_ratio']].head(10))

print(f"\nTotal body features: {len(body_features)}")

# Verify URL feature consistency
print("\nURL Feature Verification:")
print(f"Existing 'urls' column (boolean): {email_df['urls'].sum()} emails with URLs")
print(f"New 'body_url_count': {(email_df['body_url_count'] > 0).sum()} emails with URLs")
print(f"Match rate: {((email_df['urls'] == (email_df['body_url_count'] > 0)).sum() / len(email_df)) * 100:.2f}%")

In [ ]:
email_df

### DROP UNNECESSARY COLUMNS

In [ ]:
print(f"\nDataset shape BEFORE dropping: {email_df.shape}")
print(f"Columns BEFORE: {email_df.columns.tolist()}\n")

# Columns to drop with reasons
columns_to_drop = [
    'sender',           # Raw sender text (acts as unique identifier, causes model to memorize
                        # specific senders instead of learning patterns, creates data leakage
                        # and bias - legitimate senders would always be classified as legitimate)

    'sender_name',      # Extracted sender name (also acts as identifier, model could memorize
                        # specific names like "John Smith" instead of learning structural patterns,
                        # poor generalization to new/unseen names)

    'sender_email',     # Extracted sender email (unique identifier per sender, causes memorization,
                        # model won't generalize to new phishing emails from different addresses,
                        # all useful patterns already captured in engineered features)

    'sender_domain',    # Raw domain text (acts as identifier with 62k+ unique values, causes
                        # memorization of specific domains, replaced by engineered features:
                        # domain_frequency, domain_entropy, domain_length, is_rare_domain, etc.
                        # which capture domain patterns without memorizing specific domains)

    'domain_tld'        # Raw TLD text (categorical with many unique values, already replaced by
                        # numerical features: tld_phishing_ratio and tld_frequency which capture
                        # TLD patterns statistically without needing the raw text)
]

# Drop columns
email_df = email_df.drop(columns=columns_to_drop)

print(f"\nDataset shape AFTER dropping: {email_df.shape}")
print(f"\nColumns AFTER: {email_df.columns.tolist()}")

# Display remaining column categories
print("\n" + "="*80)
print("REMAINING COLUMNS BY CATEGORY")
print("="*80)

# Original columns
original_cols = ['subject', 'body', 'label', 'urls']
print(f"\nOriginal Columns ({len([c for c in original_cols if c in email_df.columns])}):")
print([c for c in original_cols if c in email_df.columns])

# Sender/Domain engineered features
sender_features = [col for col in email_df.columns if col.startswith(('sender_', 'email_', 'domain_', 'is_rare', 'tld_'))]
print(f"\nSender/Domain Engineered Features ({len(sender_features)}):")
print(sender_features)

# Subject engineered features
subject_features = [col for col in email_df.columns if col.startswith('subject_')]
print(f"\nSubject Engineered Features ({len(subject_features)}):")
print(subject_features)

# Body engineered features
body_features = [col for col in email_df.columns if col.startswith('body_')]
print(f"\nBody Engineered Features ({len(body_features)}):")
print(body_features)

print("\n" + "="*80)
print(f"TOTAL COLUMNS: {len(email_df.columns)}")
print(f"TOTAL ENGINEERED FEATURES: {len(sender_features) + len(subject_features) + len(body_features)}")
print("="*80)

## 2. Data Preprocessing

### TEXT CLEANING (After Feature Extraction, Before Vectorization)
#### Cleaning Strategy:
1. Remove URLs (features already extracted: body_url_count, body_url_density)
2. Remove ! and ? (features already extracted: exclamation_count, question_count)
3. Preserve code integrity (code patterns remain intact)
4. Clean excessive whitespace


In [ ]:
"""
STEP 1: REMOVE URLs
Reason: URL presence and density already captured in features:
  - body_url_count: actual count of URLs
  - body_url_density: URLs per 100 words
  - urls: boolean flag for URL presence
"""
# Combine all URL patterns into one
combined_url_pattern = '|'.join(url_patterns)

# Count URLs before removal (for verification)
subject_urls_before = email_df['subject'].str.contains(combined_url_pattern, regex=True).sum()
body_urls_before = email_df['body'].str.contains(combined_url_pattern, regex=True).sum()

print(f"\nURLs detected before removal:")
print(f"   Subject: {subject_urls_before} emails contain URLs")
print(f"   Body: {body_urls_before} emails contain URLs")

# Remove URLs from subject and body
email_df['subject'] = email_df['subject'].str.replace(combined_url_pattern, '', regex=True)
email_df['body'] = email_df['body'].str.replace(combined_url_pattern, '', regex=True)

# Verify removal
subject_urls_after = email_df['subject'].str.contains(combined_url_pattern, regex=True).sum()
body_urls_after = email_df['body'].str.contains(combined_url_pattern, regex=True).sum()

print(f"\nURLs detected after removal:")
print(f"   Subject: {subject_urls_after} emails contain URLs")
print(f"   Body: {body_urls_after} emails contain URLs")



"""
STEP 2: REMOVE ! AND ?
Reason: Urgency signals already captured in features:
    - subject_exclamation_count: count of ! in subject
    - subject_question_count: count of ? in subject
    - body_exclamation_count: count of ! in body
    - body_question_count: count of ? in body
    - body_exclamation_density: ! per 100 characters
"""
# Count before removal
subject_punct_before = (email_df['subject'].str.contains(r'[!?]', regex=True)).sum()
body_punct_before = (email_df['body'].str.contains(r'[!?]', regex=True)).sum()

print(f"\nEmails with ! or ? before removal:")
print(f"   Subject: {subject_punct_before} emails")
print(f"   Body: {body_punct_before} emails")

# Remove ! and ? (single or multiple occurrences)
email_df['subject'] = email_df['subject'].str.replace(r'[!?]+', '', regex=True)
email_df['body'] = email_df['body'].str.replace(r'[!?]+', '', regex=True)

# Verify removal
subject_punct_after = (email_df['subject'].str.contains(r'[!?]', regex=True)).sum()
body_punct_after = (email_df['body'].str.contains(r'[!?]', regex=True)).sum()

print(f"\nEmails with ! or ? after removal:")
print(f"   Subject: {subject_punct_after} emails")
print(f"   Body: {body_punct_after} emails")



"""
STEP 3: CLEAN EXCESSIVE WHITESPACE
"""
# Replace multiple spaces with single space and strip
email_df['subject'] = email_df['subject'].str.replace(r'\s+', ' ', regex=True).str.strip()
email_df['body'] = email_df['body'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
email_df

In [ ]:
email_df.isnull().sum()

#### CHECK AND REMOVE ALL NaN VALUES FROM DATASET

In [ ]:
print(f"\nDataset shape BEFORE removing NaNs: {email_df.shape}")

# Check for NaN values in all columns
print("\n1. NaN COUNT PER COLUMN:")
print("-"*80)
nan_summary = email_df.isnull().sum()
print(nan_summary)

total_nans = nan_summary.sum()
print(f"\nTotal NaN values across all columns: {total_nans}")

# Identify columns with NaN values
columns_with_nan = nan_summary[nan_summary > 0].index.tolist()
print(f"\nColumns with NaN values: {columns_with_nan}")

# Show percentage of NaN per column
if len(columns_with_nan) > 0:
    print("\n2. NaN PERCENTAGE PER COLUMN:")
    print("-"*80)
    for col in columns_with_nan:
        nan_pct = (email_df[col].isnull().sum() / len(email_df)) * 100
        print(f"   {col}: {email_df[col].isnull().sum()} NaNs ({nan_pct:.2f}%)")

# Count rows with ANY NaN value
rows_with_nan = email_df.isnull().any(axis=1).sum()
print(f"\n3. ROWS WITH ANY NaN VALUE:")
print("-"*80)
print(f"Total rows with NaN: {rows_with_nan}")
print(f"Percentage: {(rows_with_nan / len(email_df)) * 100:.2f}%")

# Show sample of rows with NaN
if rows_with_nan > 0:
    print("\n4. SAMPLE ROWS WITH NaN VALUES:")
    print("-"*80)
    nan_rows = email_df[email_df.isnull().any(axis=1)]
    print(nan_rows.head(10))

    # Check label distribution
    print("\n5. LABEL DISTRIBUTION OF ROWS WITH NaN:")
    print("-"*80)
    print(nan_rows['label'].value_counts())
    print(f"Phishing: {(nan_rows['label'] == 1).sum()}")
    print(f"Legitimate: {(nan_rows['label'] == 0).sum()}")

# Drop all rows with ANY NaN values
print("\n6. DROPPING ROWS WITH NaN VALUES...")
print("-"*80)
email_df_clean = email_df.dropna()

print(f"Rows dropped: {len(email_df) - len(email_df_clean)}")
print(f"Dataset shape AFTER removing NaNs: {email_df_clean.shape}")

# Update the main dataframe
email_df = email_df_clean.copy()

# Verify no NaN values remain
print("\n7.VERIFICATION - NaN COUNT AFTER CLEANING:")
print("-"*80)
remaining_nans = email_df.isnull().sum().sum()
print(f"Total remaining NaN values: {remaining_nans}")


#### SUBJECT VECTORIZATION (Unigrams Only)

In [ ]:
# Initialize Subject TF-IDF Vectorizer
subject_vectorizer = TfidfVectorizer(
    max_features=2000,      # Keep top 2000 features (subjects are short)
    ngram_range=(1, 1),     # Unigrams only (single words)
    min_df=2,               # Ignore terms appearing in less than 2 documents (noise reduction)
    max_df=0.95,            # Ignore terms appearing in more than 95% of documents (too common)
    sublinear_tf=True       # Use log scaling for term frequency (reduces impact of very frequent terms)
)

# Fit and transform subject text
subject_tfidf = subject_vectorizer.fit_transform(email_df['subject'])

print(f"Shape: {subject_tfidf.shape}")
print(f"Features: {subject_tfidf.shape[1]} TF-IDF features (unigrams)")
print(f"Sparsity: {(1 - subject_tfidf.nnz / (subject_tfidf.shape[0] * subject_tfidf.shape[1])) * 100:.2f}%")

# Sample feature names
print(f"\nSample features (first 20):")
print(f"{subject_vectorizer.get_feature_names_out()[:20]}")

# Convert subject TF-IDF to DataFrame
subject_tfidf_df = pd.DataFrame(
    subject_tfidf.toarray(),
    columns=[f'subject_tfidf_{i}' for i in range(subject_tfidf.shape[1])],
    index=email_df.index  # Preserve original index
)

print(f"Subject TF-IDF DataFrame created: {subject_tfidf_df.shape}")

# Combine with email_df (drop original subject column)
email_df = email_df.drop(columns=['subject'])
email_df = pd.concat([email_df, subject_tfidf_df], axis=1)

print(f"Subject column replaced with TF-IDF features!")
print(f"Updated email_df shape: {email_df.shape}")

#### BODY VECTORIZATION (Unigrams + Bigrams)

##### Initialize Body TF-IDF Vectorizer

##### We use BOTH unigrams and bigrams for body because:

1. Phishing emails use specific PHRASES that indicate malicious intent:
    - "click here", "verify account", "limited time", "act now", "suspended account"
2. Single words (unigrams) alone miss context:
    - "verify" vs "verify account" - the phrase is more suspicious
    - "click" vs "click here" - the phrase is a stronger phishing signal
3. Body text is long enough (~1000 chars avg) to have meaningful bigrams
4. Bigrams capture common phishing patterns that appear as consecutive words
5. The combination gives the model both individual keywords AND contextual phrases

In [ ]:
body_vectorizer = TfidfVectorizer(
    max_features=5000,      # Keep top 5000 features (body is longer, needs more features)
    ngram_range=(1, 2),     # Both unigrams (1) and bigrams (2) - captures phrases!
    min_df=2,               # Ignore terms appearing in less than 2 documents
    max_df=0.95,            # Ignore terms appearing in more than 95% of documents
    sublinear_tf=True       # Use log scaling for term frequency
)

# Fit and transform body text
body_tfidf = body_vectorizer.fit_transform(email_df['body'])

print(f"Shape: {body_tfidf.shape}")
print(f"Features: {body_tfidf.shape[1]} TF-IDF features (unigrams + bigrams)")
print(f"Sparsity: {(1 - body_tfidf.nnz / (body_tfidf.shape[0] * body_tfidf.shape[1])) * 100:.2f}%")

# Sample feature names
print(f"\nSample features (first 20):")
sample_features = body_vectorizer.get_feature_names_out()[:20]
print(f"{sample_features}")

# Analyze unigrams vs bigrams
all_features = body_vectorizer.get_feature_names_out()
unigrams = [f for f in all_features if ' ' not in f]
bigrams = [f for f in all_features if ' ' in f]

print(f"\nUnigrams count: {len(unigrams)}")
print(f"Bigrams count: {len(bigrams)}")
print(f"\nSample unigrams (first 10): {unigrams[:10]}")
print(f"Sample bigrams (first 10): {bigrams[:10]}")

# Convert body TF-IDF to DataFrame
body_tfidf_df = pd.DataFrame(
    body_tfidf.toarray(),
    columns=[f'body_tfidf_{i}' for i in range(body_tfidf.shape[1])],
    index=email_df.index  # Preserve original index
)

print(f"Body TF-IDF DataFrame created: {body_tfidf_df.shape}")

# Combine with email_df (drop original body column)
email_df = email_df.drop(columns=['body'])
email_df = pd.concat([email_df, body_tfidf_df], axis=1)

print(f"Body column replaced with TF-IDF features!")
print(f"Updated email_df shape: {email_df.shape}")

#### SAVE VECTORIZERS FOR INFERENCE

In [ ]:
# Create directory
os.makedirs('models/pipeline_components', exist_ok=True)

# Save the fitted vectorizers
joblib.dump(subject_vectorizer, 'models/pipeline_components/subject_vectorizer.pkl')
joblib.dump(body_vectorizer, 'models/pipeline_components/body_vectorizer.pkl')

In [ ]:
email_df

## Feature Analysis & Selection

#### Multi-stage approach:

1. Remove useless features: Zero/near-zero variance, missing >50%
2. Statistical significance: Correlation + Mutual Information + ANOVA F-test with label
3. Remove redundancy: High feature-to-feature correlations (keep best one)
4. Validate with tree model: Quick Random Forest to see importance rankings
5. Final selection: Top 15-20 features based on combined evidence

##### STEP 1: Remove Low Variance Features

In [ ]:
# STEP 1: Remove Useless Features (Zero/Near-Zero Variance)
# All 35 engineered features
all_engineered_features = ['sender_missing', 'sender_name_length', 'email_local_length',
                           'domain_length', 'domain_has_numbers', 'sender_is_noreply',
                           'consecutive_digit_length', 'special_char_count', 'domain_entropy',
                           'domain_vowel_ratio', 'domain_consonant_ratio', 'domain_frequency',
                           'is_rare_domain', 'tld_frequency', 'tld_phishing_ratio',
                           'subject_length', 'subject_word_count', 'subject_missing',
                           'subject_exclamation_count', 'subject_question_count',
                           'subject_special_char_ratio', 'subject_avg_word_length', 'subject_entropy',
                           'body_length', 'body_word_count', 'body_to_subject_length_ratio',
                           'body_url_count', 'body_url_density', 'body_exclamation_count',
                           'body_exclamation_density', 'body_question_count', 'body_avg_word_length',
                           'body_entropy', 'body_unique_word_ratio', 'body_line_count']

print("="*80)
print("STEP 1: VARIANCE ANALYSIS")
print("="*80)

print(f"Starting with: {len(all_engineered_features)} features\n")

# Calculate variance
variance = email_df[all_engineered_features].var()

# Find zero/near-zero variance features (threshold: 0.01)
low_variance_features = variance[variance < 0.01].index.tolist()

print("Low Variance Features (variance < 0.01):")
if low_variance_features:
    for feat in low_variance_features:
        print(f"  - {feat}: variance = {variance[feat]:.6f}")
else:
    print("  None found")

# Keep features with sufficient variance
step1_features = [f for f in all_engineered_features if f not in low_variance_features]

print(f"\nKept: {len(step1_features)} features")
print(f"Removed: {len(low_variance_features)} features")
print(f"\nRemaining features: {step1_features}")

##### STEP 2: Statistical Significance Tests

In [ ]:
# STEP 2: Statistical Significance (Correlation, Mutual Info, ANOVA F-test)
# Prepare data
X = email_df[step1_features]
y = email_df['label']

print(f"Analyzing {len(step1_features)} features\n")

# Initialize results storage
results = []

# -------------------------------------------------------------------------
# 2.1: Pearson Correlation with Label
# -------------------------------------------------------------------------
correlations = X.corrwith(y).abs()

# -------------------------------------------------------------------------
# 2.2: Mutual Information (captures non-linear relationships)
# -------------------------------------------------------------------------
mi_scores = mutual_info_classif(X, y, random_state=42)

# -------------------------------------------------------------------------
# 2.3: ANOVA F-test
# -------------------------------------------------------------------------
f_scores, p_values = f_classif(X, y)

# -------------------------------------------------------------------------
# 2.4: Combine all metrics
# -------------------------------------------------------------------------
for i, feature in enumerate(step1_features):
    results.append({
        'Feature': feature,
        'Correlation': correlations[feature],
        'Mutual_Info': mi_scores[i],
        'F_Score': f_scores[i],
        'P_Value': p_values[i]
    })

# Create results dataframe
results_df = pd.DataFrame(results)

# Calculate composite score (normalized average of all metrics)
scaler = MinMaxScaler()

results_df['Corr_Norm'] = scaler.fit_transform(results_df[['Correlation']])
results_df['MI_Norm'] = scaler.fit_transform(results_df[['Mutual_Info']])
results_df['F_Norm'] = scaler.fit_transform(results_df[['F_Score']])

# Composite score: average of normalized metrics
results_df['Composite_Score'] = (results_df['Corr_Norm'] +
                                  results_df['MI_Norm'] +
                                  results_df['F_Norm']) / 3

# Sort by composite score
results_df = results_df.sort_values('Composite_Score', ascending=False)

print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE RESULTS")
print("="*80)

print("\nTop 20 Features by Composite Score:")
print(results_df[['Feature', 'Correlation', 'Mutual_Info', 'F_Score',
                  'Composite_Score']].head(20).to_string(index=False))

print("\n" + "="*80)
print("Full Rankings:")
print(results_df[['Feature', 'Composite_Score']].to_string(index=False))

# Save for next step
step2_results = results_df
print(f"\nStep 2 complete! Features ranked by statistical significance.")

##### STEP 3: Remove Redundant Features

In [ ]:
# STEP 3: Remove Redundant Features (High Correlation)

# Calculate correlation matrix
correlation_matrix = email_df[step1_features].corr().abs()

print(f"Analyzing correlations between {len(step1_features)} features\n")

# Find highly correlated pairs (threshold: 0.85)
corr_threshold = 0.85
high_corr_pairs = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if correlation_matrix.iloc[i, j] > corr_threshold:
            feat1 = correlation_matrix.columns[i]
            feat2 = correlation_matrix.columns[j]
            high_corr_pairs.append({
                'Feature_1': feat1,
                'Feature_2': feat2,
                'Correlation': correlation_matrix.iloc[i, j]
            })

print(f"Found {len(high_corr_pairs)} highly correlated pairs (>0.85):\n")

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)
    print(high_corr_df.to_string(index=False))

    # For each pair, remove the feature with lower composite score from Step 2
    features_to_remove = set()

    print("\n" + "="*80)
    print("REDUNDANCY RESOLUTION (keeping feature with higher composite score):")
    print("="*80 + "\n")

    for _, row in high_corr_df.iterrows():
        feat1, feat2 = row['Feature_1'], row['Feature_2']

        # Skip if already marked for removal
        if feat1 in features_to_remove or feat2 in features_to_remove:
            continue

        # Get composite scores from Step 2
        score1 = step2_results[step2_results['Feature'] == feat1]['Composite_Score'].values[0]
        score2 = step2_results[step2_results['Feature'] == feat2]['Composite_Score'].values[0]

        # Remove the one with lower score
        if score1 > score2:
            features_to_remove.add(feat2)
            print(f"Keep: {feat1} (score: {score1:.4f})")
            print(f"Remove: {feat2} (score: {score2:.4f})")
            print(f"Correlation: {row['Correlation']:.3f}\n")
        else:
            features_to_remove.add(feat1)
            print(f"Keep: {feat2} (score: {score2:.4f})")
            print(f"Remove: {feat1} (score: {score1:.4f})")
            print(f"Correlation: {row['Correlation']:.3f}\n")

    # Remove redundant features
    step3_features = [f for f in step1_features if f not in features_to_remove]

    print("="*80)
    print(f"Removed {len(features_to_remove)} redundant features:")
    for feat in features_to_remove:
        print(f"  - {feat}")

else:
    print("No highly correlated pairs found!")
    step3_features = step1_features

print(f"\nKept: {len(step3_features)} features")
print(f"\nRemaining features: {step3_features}")

##### STEP 4: Random Forest Feature Importance

In [ ]:
# STEP 4: Validate with Random Forest Feature Importance

# Prepare data
X = email_df[step3_features]
y = email_df['label']

print(f"Training Random Forest on {len(step3_features)} features...\n")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      random_state=42, stratify=y)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                            max_depth=10)
rf.fit(X_train, y_train)

# Get feature importances
importances = rf.feature_importances_

# Create importance dataframe
rf_importance_df = pd.DataFrame({
    'Feature': step3_features,
    'RF_Importance': importances
}).sort_values('RF_Importance', ascending=False)

# Normalize importance scores
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
rf_importance_df['RF_Importance_Norm'] = scaler.fit_transform(
    rf_importance_df[['RF_Importance']]
)

print("Random Forest Feature Importance's (Top 20):")
print(rf_importance_df.head(20).to_string(index=False))

# Calculate model performance
train_score = rf.score(X_train, y_train)
test_score = rf.score(X_test, y_test)

print(f"\nModel Performance:")
print(f"Training Accuracy: {train_score:.4f}")
print(f"Test Accuracy: {test_score:.4f}")

# Save for next step
step4_results = rf_importance_df
print(f"\nStep 4 complete! Feature importance calculated.")

##### STEP 5: Final Feature Selection

In [ ]:
# STEP 5: Final Selection - Top 15-20 Features

# Merge statistical results (Step 2) with RF importance (Step 4)
final_results = step2_results[step2_results['Feature'].isin(step3_features)].copy()
final_results = final_results.merge(step4_results[['Feature', 'RF_Importance_Norm']],
                                     on='Feature', how='left')

# Calculate final composite score (equal weight: stats + RF)
final_results['Final_Score'] = (final_results['Composite_Score'] +
                                 final_results['RF_Importance_Norm']) / 2

# Sort by final score
final_results = final_results.sort_values('Final_Score', ascending=False)

print("\n" + "="*80)
print("COMBINED RANKINGS (Statistical + Random Forest)")
print("="*80 + "\n")

print("All Features Ranked:")
print(final_results[['Feature', 'Composite_Score', 'RF_Importance_Norm',
                     'Final_Score']].to_string(index=False))

# Select top 15 features
top_n = 15
selected_features = final_results.head(top_n)['Feature'].tolist()

print("\n" + "="*80)
print(f"TOP {top_n} SELECTED FEATURES")
print("="*80 + "\n")

for i, row in final_results.head(top_n).iterrows():
    print(f"{final_results.head(top_n).index.tolist().index(i)+1:2d}. {row['Feature']:30s} "
          f"(Final Score: {row['Final_Score']:.4f})")

print("\n" + "="*80)
print("FEATURE SELECTION SUMMARY")
print("="*80)

print(f"""
Started with: {len(all_engineered_features)} features
After Step 1: {len(step1_features)} features (removed low variance)
After Step 3: {len(step3_features)} features (removed redundant)
Final Selection: {len(selected_features)} features

Reduction: {len(all_engineered_features)} → {len(selected_features)} features
({(1 - len(selected_features)/len(all_engineered_features))*100:.1f}% reduction)
""")

# Show selected features by category
subject_selected = [f for f in selected_features if f.startswith('subject_')]
body_selected = [f for f in selected_features if f.startswith('body_')]
sender_selected = [f for f in selected_features if not f.startswith('subject_')
                   and not f.startswith('body_')]

print("Selected Features by Category:")
print(f"Subject: {len(subject_selected)} - {subject_selected}")
print(f"Body: {len(body_selected)} - {body_selected}")
print(f"Sender: {len(sender_selected)} - {sender_selected}")

print("\n" + "="*80)
print("FINAL SELECTED FEATURES (copy for next steps):")
print("="*80)
print(f"\nfinal_selected_features = {selected_features}")

#### VISUALIZATIONS - Correlation Analysis of Final 15 Features

In [ ]:
# Final selected features from Step 5
final_selected_features = [
    'domain_frequency', 'is_rare_domain', 'body_entropy',
    'tld_phishing_ratio', 'body_unique_word_ratio',
    'body_exclamation_density', 'body_word_count',
    'body_to_subject_length_ratio', 'email_local_length',
    'body_url_density', 'consecutive_digit_length',
    'body_exclamation_count', 'subject_entropy',
    'body_url_count', 'tld_frequency'
]

print("="*80)
print("STEP 6: CORRELATION VISUALIZATIONS")
print("="*80)
print(f"\nVisualizing {len(final_selected_features)} final selected features\n")

# VISUALIZATION 1: Feature-to-Feature Correlation Heatmap

# Calculate correlation matrix
correlation_matrix = email_df[final_selected_features].corr()

# Create figure
plt.figure(figsize=(16, 14))

# Create heatmap
sns.heatmap(correlation_matrix,
            annot=True,  # Show correlation values
            fmt='.2f',   # Format to 2 decimal places
            cmap='coolwarm',  # Color scheme (blue=negative, red=positive)
            center=0,    # Center colormap at 0
            square=True, # Make cells square
            linewidths=0.5,
            cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1)

plt.title('Feature-to-Feature Correlation Matrix\n(Final 15 Selected Features)',
          fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()


# VISUALIZATION 2: Feature-to-Label Correlation (Bar Chart)

# Calculate correlation with label
label_correlations = email_df[final_selected_features + ['label']].corr()['label'].drop('label')
label_correlations = label_correlations.sort_values(ascending=True)

# Create figure
plt.figure(figsize=(12, 8))

# Create horizontal bar chart
colors = ['red' if x < 0 else 'green' for x in label_correlations.values]
plt.barh(range(len(label_correlations)), label_correlations.values, color=colors, alpha=0.7)

plt.yticks(range(len(label_correlations)), label_correlations.index, fontsize=10)
plt.xlabel('Correlation with Label (Phishing)', fontsize=12, fontweight='bold')
plt.title('Feature Correlation with Target Label\n(Positive = Higher in Phishing Emails)',
          fontsize=14, fontweight='bold', pad=15)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(label_correlations.values):
    plt.text(v + 0.01 if v > 0 else v - 0.01, i, f'{v:.3f}',
             va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


# VISUALIZATION 3: Correlation Strength Summary
# Get upper triangle of correlation matrix (to avoid duplicates)
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Flatten and remove NaN values
correlations_flat = upper_triangle.stack().abs()

# Create histogram
plt.figure(figsize=(10, 6))
plt.hist(correlations_flat, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
plt.axvline(x=0.85, color='red', linestyle='--', linewidth=2,
            label='Redundancy Threshold (0.85)')
plt.xlabel('Absolute Correlation Value', fontsize=12, fontweight='bold')
plt.ylabel('Frequency', fontsize=12, fontweight='bold')
plt.title('Distribution of Feature-to-Feature Correlations\n(Final 15 Features)',
          fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


# VISUALIZATION 4: Feature Importance vs Correlation with Label

# Merge final scores with label correlations
viz_data = final_results[final_results['Feature'].isin(final_selected_features)].copy()
viz_data['Label_Correlation'] = viz_data['Feature'].map(
    lambda x: email_df[final_selected_features + ['label']].corr()['label'][x]
).abs()

# Add numbered index for reference
viz_data = viz_data.sort_values('Final_Score', ascending=False).reset_index(drop=True)
viz_data['ID'] = range(1, len(viz_data) + 1)

# Create figure with two subplots side by side
fig = plt.figure(figsize=(20, 8))

# Left subplot: Scatter plot with numbers only
ax1 = plt.subplot(1, 2, 1)

scatter = ax1.scatter(viz_data['Label_Correlation'],
                      viz_data['Final_Score'],
                      s=400,
                      c=viz_data['Final_Score'],
                      cmap='viridis',
                      alpha=0.8,
                      edgecolors='black',
                      linewidths=2.5)

# Add number labels on each point (much cleaner!)
for idx, row in viz_data.iterrows():
    ax1.text(row['Label_Correlation'], row['Final_Score'],
             str(row['ID']),
             fontsize=12,
             fontweight='bold',
             ha='center',
             va='center',
             color='white',
             bbox=dict(boxstyle='circle,pad=0.1',
                      facecolor='red',
                      edgecolor='white',
                      alpha=0.9,
                      linewidth=2))

ax1.set_xlabel('Correlation with Label (Absolute)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Final Importance Score', fontsize=13, fontweight='bold')
ax1.set_title('Feature Importance vs Label Correlation\n(See legend on right for feature names)',
              fontsize=14, fontweight='bold', pad=15)
ax1.grid(alpha=0.3, linestyle='--')
plt.colorbar(scatter, ax=ax1, label='Final Score', shrink=0.8)

# Right subplot: Legend table with feature names
ax2 = plt.subplot(1, 2, 2)
ax2.axis('off')

# Create table data
table_data = []
for idx, row in viz_data.iterrows():
    table_data.append([
        f"{row['ID']}",
        row['Feature'],
        f"{row['Final_Score']:.3f}",
        f"{row['Label_Correlation']:.3f}"
    ])

# Create table
table = ax2.table(cellText=table_data,
                  colLabels=['#', 'Feature Name', 'Final Score', 'Label Corr'],
                  cellLoc='left',
                  loc='center',
                  colWidths=[0.08, 0.50, 0.20, 0.20])

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Style header
for i in range(4):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Color rows by importance (gradient)
colors = plt.cm.YlGn(np.linspace(0.3, 0.9, len(viz_data)))
for i in range(len(viz_data)):
    for j in range(4):
        table[(i+1, j)].set_facecolor(colors[i])
        table[(i+1, j)].set_alpha(0.7)

ax2.set_title('Feature Reference Table\n(Ranked by Final Score)',
              fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()


# SUMMARY STATISTICS
print("="*80)
print("CORRELATION ANALYSIS SUMMARY")
print("="*80)

print(f"\nFeature-to-Feature Correlations:")
print(f"  • Maximum correlation: {correlations_flat.max():.3f}")
print(f"  • Average correlation: {correlations_flat.mean():.3f}")
print(f"  • Correlations > 0.5: {(correlations_flat > 0.5).sum()}")
print(f"  • Correlations > 0.7: {(correlations_flat > 0.7).sum()}")
print(f"  • Correlations > 0.85: {(correlations_flat > 0.85).sum()} (Should be 0!)")

print(f"\nFeature-to-Label Correlations:")
print(f"  • Strongest positive: {label_correlations.idxmax()} ({label_correlations.max():.3f})")
print(f"  • Strongest negative: {label_correlations.idxmin()} ({label_correlations.min():.3f})")
print(f"  • Average absolute: {label_correlations.abs().mean():.3f}")
print("="*80)

## 3. Combine the selected features and the TF-IDF feature
* Where no longer splitting the data to train and

In [ ]:
import pandas as pd

# Selected 15 engineered features
X_engineered = email_df[selected_features]

# All TF-IDF features (7000 columns)
tfidf_cols = [col for col in email_df.columns if 'tfidf' in col]
X_tfidf = email_df[tfidf_cols]

# Combine TF-IDF + Engineered Features + Label
pac_final_data = pd.concat([X_tfidf, X_engineered, email_df[['label']]], axis=1)
pac_final_data

In [ ]:
# Export Training Data to CSV
import os
# Create directory
os.makedirs('datasets/training', exist_ok=True)

# Export
output_path = 'datasets/training/final_dataset.csv'
pac_final_data.to_csv(output_path, index=False)

print(f"\nDataset exported to: {output_path}")
print(f"Shape: {pac_final_data.shape}")
print(f"Total columns: {pac_final_data.shape[1]} (7015 features + 1 label)")
